In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/netflix-shows")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'netflix-shows' dataset.
Path to dataset files: /kaggle/input/netflix-shows


In [5]:
files = os.listdir(path)
print(files)

['netflix_titles.csv']


In [6]:
df = pd.read_csv(os.path.join(path, "netflix_titles.csv"))
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [7]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added'] = df['date_added'].dt.year

In [8]:
type_cnt = df['type'].value_counts().reset_index()
type_cnt.columns = ['Type', 'Count']

fig = px.pie(type_cnt, values='Count', names='Type',
             title='Netflix-ի ֆիլմեր և սերիալների բաշխումը',
             hole=0.5,
             color_discrete_sequence=['#E51914', '#221E1F'])
fig.update_traces(textinfo='percent+label')
fig.show()

In [9]:
content_by_year = df.groupby(['year_added', 'type']).size().reset_index(name='count')

fig = px.area(content_by_year, x='year_added', y='count', color='type',
              title='Ֆիլմերի և սերիալների աճը տարիների ընթացքում',
              labels={'year_added': 'Տարի', 'count': 'Քանակ', 'type': 'Տեսակ'},
              color_discrete_map={'Movie': '#E50914', 'TV Show': '#221F1F'})
fig.show()

In [10]:
yearly_movies = df[df['type'] == 'Movie'].groupby('year_added').size()
yearly_tv = df[df['type'] == 'TV Show'].groupby('year_added').size()

line1 = go.Scatter(x=yearly_movies.index, y=yearly_movies.values,
                          mode='lines+markers', name='Movies', line=dict(color='#E50914'))
line2 = go.Scatter(x=yearly_tv.index, y=yearly_tv.values,
                          mode='lines+markers', name='TV Shows', line=dict(color='#221F1F'))

fig_lines = go.Figure(data=[line1, line2])
fig_lines.update_layout(title="Ֆիլմերի և սերիալների աճը տարիների ընթացքում",
                        xaxis_title="Տարի", yaxis_title="Քանակ")
fig_lines.show()

In [11]:
genres = df.assign(listed_in=df['listed_in'].str.split(', ')).explode('listed_in')

fig = px.treemap(genres, path=['type', 'listed_in'],
                 title='Content-ի հիերարխիան ըստ տեսակի և ժանրի',
                 color_discrete_sequence=['#E43914', '#C0C0C0'])
fig.show()

In [12]:
top_countries = df['country'].str.split(', ').explode().value_counts().head(10).reset_index()
top_countries.columns = ['Country', 'Count']

fig = px.bar(top_countries,
             x='Count',
             y='Country',
             orientation='h',
             title='Թոփ 10 երկրներն ըստ content֊ի քանակի',
             labels={'Count': 'Քանակ', 'Country': 'Երկիր'},
             color = 'Count',
             text='Count',
             color_continuous_scale='Reds')

fig.update_traces(textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'},
                  coloraxis_showscale=False,
                  template='plotly_white')
fig.show()

In [13]:
df_map = df.groupby(['year_added', 'country']).size().reset_index(name='count')

fig = px.choropleth(df_map,
                    locations="country",
                    locationmode='country names',
                    color="count",
                    hover_name="country",
                    animation_frame="year_added",
                    color_continuous_scale=px.colors.sequential.Reds,
                    title="Netflix-ի գլոբալ տարածումը")
fig.show()

In [14]:
df_movies = df[df['type'] == 'Movie'].copy()
df_movies['duration_min'] = df_movies['duration'].str.replace(' min', '').astype(float)

fig = px.histogram(df_movies, x='duration_min', nbins=50,
                   title='Ֆիլմերի տևողության բաշխվածությունը (րոպե)',
                   labels={'duration_min': 'Տևողություն (րոպե)'},
                   color_discrete_sequence=['#ff3333'])

fig.update_layout(yaxis_title="Քանակ",
                  bargap=0.1,
                  template='plotly_white')
fig.show()

In [15]:
rating_cnt = df.groupby(['rating', 'type']).size().unstack(fill_value=0).reset_index()
rating_cnt = rating_cnt.sort_values(by='Movie', ascending=False).head(10)

labels = rating_cnt['rating']
values_movies = rating_cnt['Movie']
values_tv = rating_cnt['TV Show']

trace1 = go.Bar(x=labels, y=values_movies, name="Movies", marker_color='#E50914')
trace2 = go.Bar(x=labels, y=values_tv, name="TV Shows", marker_color='#221F1F')

data = [trace1, trace2]

layout = go.Layout(
    title="Ֆիլմերի և սերիալների համեմատությունը ըստ Rating֊ի",
    barmode='group',
    xaxis_title="Rating",
    yaxis_title="Քանակ"
)

fig = go.Figure(data=data, layout=layout)
fig.show()

In [16]:
fig = px.sunburst(df.dropna(subset=['rating', 'type']),
                  path=['type', 'rating'],
                  values=None,
                  title="Netflix-ի content֊ի հիերարխիան",
                  color='type',
                  color_discrete_map={'Movie':'#E50914', 'TV Show':'#221F1F'})
fig.show()

In [17]:
df_movies = df[df['type'] == 'Movie'].copy()
df_movies['duration_min'] = df_movies['duration'].str.replace(' min', '').astype(float)

df_movies_genres = df_movies.assign(genre=df_movies['listed_in'].str.split(', ')).explode('genre')

top_genres = df_movies_genres['genre'].value_counts().head(10).index
df_top_genres = df_movies_genres[df_movies_genres['genre'].isin(top_genres)]

fig = px.box(df_top_genres,
             x="genre",
             y="duration_min",
             color="genre",
             points="outliers",
             template="simple_white",
             title="Ֆիլմերի տևողության բաշխվածությունը ըստ թոփ ժանրերի",
             labels={"genre": "Ժանր", "duration_min": "Տևողություն (րոպե)"})

fig.update_layout(showlegend=False)
fig.show()

In [18]:
df_tv = df[df['type'] == 'TV Show'].copy()
df_tv['seasons'] = df_tv['duration'].str.replace(' Seasons', '').str.replace(' Season', '').astype(int)

df_tv_genres = df_tv.assign(genre=df_tv['listed_in'].str.split(', ')).explode('genre')

avg_seasons = df_tv_genres.groupby('genre')['seasons'].mean().sort_values(ascending=False).head(15).reset_index()

fig = go.Figure(go.Bar(
    x=avg_seasons['genre'],
    y=avg_seasons['seasons'],
    marker_color='#E50914',
    text=avg_seasons['seasons'].round(2),
    textposition='auto',
))

fig.update_layout(
    title="Սեզոնների միջին քանակը ըստ ժանրերի (Թոփ 15)",
    xaxis_title="Ժանր",
    yaxis_title="Սեզոնների միջին քանակ",
    template="plotly_white",
    xaxis={'categoryorder':'total descending'}
)

fig.show()